# Jet Bigrams — GPT-2 Large

Evaluates the jet bigram expansions over the full vocabulary:

- **E → U**: pure embedding-decoder path (`embedding_decoder`)
- **E → MLP$_l$ → U**: embedding through one MLP layer then decoder (`embedding_mlp_decoder`)

For each input token $z_1$, the bigram path computes the conditional likelihood of token $z_2$, for all tokens in the vocabulary

In [1]:
import torch
import jex.models as models
from jex.ngrams.bigrams import embedding_decoder, embedding_mlp_decoder
from jex.ngrams.utils import (
    eval_over_vocab,
    decode_topk,
    print_bigram_tables_side_by_side,
)

## Load model

In [2]:
device = "cpu"
if torch.backends.mps.is_available():
    device = "mps"
if torch.cuda.is_available():
    device = "cuda"
print(f"Using {device}")
lm = models.from_pretrained("gpt2-large")
lm.model.to(device)
print(f"Loaded {lm.name}: {lm.depth} layers, vocab size {lm.vocab_size}")

Using mps


Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

Loaded gpt2-large: 36 layers, vocab size 50257


## Evaluate over the full vocabulary

We run each bigram expansion on every token in the vocabulary (in batches) and collect the global top-n bigrams directly.

In [3]:
n = 100

jet_eu = embedding_decoder(lm)
with torch.no_grad():
    topk_eu = decode_topk(*eval_over_vocab(jet_eu, lm, global_topk=n), lm)
print("E→U done")

E→U done


In [4]:
topk_mlp = {}
for l in set([lm.depth // 4, lm.depth // 2, lm.depth - 1]):
    jet = embedding_mlp_decoder(lm, l)
    with torch.no_grad():
        topk_mlp[l] = decode_topk(*eval_over_vocab(jet, lm, global_topk=n), lm)
    print(f"layer {l} done", end="\r")
print("All done.")

All done.done


## Top-100 bigram pairs

Global top-100 (input, next) pairs by probability score.

In [5]:
print_bigram_tables_side_by_side(
    [("E → U", topk_eu), *[(f"E → MLP_{l} → U", rows) for l, rows in topk_mlp.items()]]
)

┌─────────┬──────┬─────────┬─────────────────┬─────────────────┬────────┬──────────────────┬─────────────────┬────────┬──────────────────┬─────────┬────────┐
│ E → U   │      │         │ E → MLP_9 → U   │                 │        │ E → MLP_18 → U   │                 │        │ E → MLP_35 → U   │         │        │
├─────────┼──────┼─────────┼─────────────────┼─────────────────┼────────┼──────────────────┼─────────────────┼────────┼──────────────────┼─────────┼────────┤
│ '#'     │ '#'  │ 100.00% │ ' protections'  │ ' Bad'          │ 76.83% │ ' realistically' │ ' ad'           │ 64.94% │ ' P'             │ ' P'    │ 89.43% │
│ '$'     │ '$'  │ 100.00% │ ' immedi'       │ ' Lag'          │ 70.95% │ ' Dar'           │ ' inf'          │ 58.87% │ ' V'             │ ' V'    │ 89.09% │
│ '&'     │ '&'  │ 100.00% │ ' buyer'        │ ' Lag'          │ 70.81% │ ' described'     │ ' pres'         │ 57.12% │ ' man'           │ 'man'   │ 83.12% │
│ "'"     │ "'"  │ 100.00% │ ' urgency'      │ ' Lag